In [37]:
import torch 
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [38]:
import seaborn as sns
iris = sns.load_dataset('iris')

In [39]:
data = torch.tensor( iris[iris.columns[0:4]].values ).float()

labels = torch.zeros(len(data), dtype=torch.long)

labels[iris.species=='versicolor'] = 1
labels[iris.species=='virinica'] = 2

In [40]:
from sklearn.model_selection import train_test_split

train_data, test_data, train_labels, test_labels = train_test_split(
    data, labels, train_size=.8
)

train_dataset = TensorDataset(train_data, train_labels)
test_dataset = TensorDataset(test_data, test_labels)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, drop_last=False)

In [41]:
def createANewModel(L2lambda):

    ANNiris = nn.Sequential(
        nn.Linear(4, 64),
        nn.ReLU(),
        nn.Linear(64, 64),
        nn.ReLU(),
        nn.Linear(64, 3)
    )

    lossfun = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(ANNiris.parameters(), lr=.005, weight_decay=L2lambda)

    return ANNiris, lossfun, optimizer

In [ ]:
numepochs = 1000

def trainTheModel():

    trainAcc = []
    testAcc = []
    losses = []
    
    # loop over epochs
    for epochi in range(numepochs):

        # ===== Train =====
        ANNiris.train()
        
        batchAcc = []
        batchLoss = []
        for X, y in train_dataloader:

            # forwardpropagation  n loss
            yHat = ANNiris(X)
            loss = lossfun(yHat, y)

            # backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # accuracy
            batchAcc.append(100* torch.mean((torch.argmax(yHat, axis=1) == y).float()).item())
            batchLoss.append( loss.item() )

        # Compute and save stats
        trainAcc.append( np.mean(batchAcc) )
        losses.append( np.mean(batchLoss) )

        # ===== Test =====
        ANNiris.eval()
        batchAccTest = []
        
        with torch.no_grad():
            for X, y in test_dataloader:
                
                yHat = ANNiris(X)
                
                predlabels = torch.argmax(yHat, axis=1)
                acc = 100*torch.mean((predlabels==y)).float()).item()
                batchAccTest.append(acc)
                
        testAcc.append(np.mean(batchAccTest))

    return trainAcc, testAcc, losses

In [ ]:
L2lambda = .01
ANNiris, lossfun, optimizer = createANewModel(L2lambda)

trainAcc, testAcc, losses = trainTheModel()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15,5))

ax[0].plot(losses, 'k⁻')
ax[0].set_ylabel('Loss')
ax[0].set_xlabe('Epochs')
ax[0].set_title('Losses with L2', str(L2lambda))

ax[1].plot(trainAcc, 'ro-')
ax[1].plot(testAcc, 'bs-')
ax[1].set_title('Accuracy with L2', str(L2lambda))
ax[1].setxlabel('Epochs')
ax[1].set_ylabel('Accyracy')
ax[1].legend(['Train', 'Test'])

plt.plot()